# 5. Local development with `DefaultAzureCredential`

The question every Azure-adjacent developer asks on day one: **"How do I run this thing on my laptop if the real auth only works in Azure?"**

Answer: `DefaultAzureCredential` from `azure-identity`. It's a **credential chain** that walks through multiple credential sources and uses the first one that works. That's why the exact same code:

```python
from azure.identity import DefaultAzureCredential
cred = DefaultAzureCredential()
token = cred.get_token('https://storage.azure.com/.default').token
```

… works on your laptop *and* in Container Apps *and* in GitHub Actions *and* in Azure Functions. No `if` branches. No `#ifdef LOCAL`.

## The credential chain (in order)

1. **EnvironmentCredential** — looks for `AZURE_CLIENT_ID` + (`AZURE_CLIENT_SECRET` or cert) + `AZURE_TENANT_ID`.
2. **WorkloadIdentityCredential** — federated tokens (AKS workload identity, GitHub Actions OIDC).
3. **ManagedIdentityCredential** — talks to IMDS — only works inside Azure compute.
4. **SharedTokenCacheCredential** — legacy, cached tokens from other MS tooling.
5. **VisualStudioCodeCredential** — the "Azure Account" extension's cached token.
6. **AzureCliCredential** — whatever `az login` gave you.
7. **AzurePowerShellCredential** — same for `Connect-AzAccount`.
8. **AzureDeveloperCliCredential** — the newer `azd` CLI.

Each one is tried in order. First success wins. You can skip expensive steps with the `exclude_*` flags if you know what environment you're in.

## The three local-dev patterns (pick one)

### Pattern A — Developer identity (recommended for humans)

```bash
az login
# then run your app
```

`DefaultAzureCredential` skips to `AzureCliCredential`, which exchanges your logged-in principal for a token. **You** call the API as **you**. Grant your user account permissions in dev.

### Pattern B — Local service principal (for reproducing prod app-only flows)

```bash
export AZURE_CLIENT_ID=<app-registration-id>
export AZURE_CLIENT_SECRET=<secret>
export AZURE_TENANT_ID=<tenant-id>
```

`EnvironmentCredential` picks these up first. Secret goes in `.env` (git-ignored) or 1Password / macOS Keychain — *not* in source.

### Pattern C — Emulator / mock (this lab)

When the real Azure dependency isn't available (or you want offline/CI speed), point the SDK at a mock that speaks the same protocols. We ship one in `fake-entra/`.

Most teams mix A + B: humans use `az login`, CI pipelines use service principals with OIDC federation (no stored secret).

In [ ]:
# Demo: watch DefaultAzureCredential's chain in action with a log handler
import logging, os
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s %(message)s')
logging.getLogger('azure.identity').setLevel(logging.DEBUG)

# Force pattern B - service principal from env
os.environ['AZURE_CLIENT_ID']     = 'daemon-client-id'
os.environ['AZURE_CLIENT_SECRET'] = 'daemon-secret-value'
os.environ['AZURE_TENANT_ID']     = 'contoso'

from azure.identity import DefaultAzureCredential
# Exclude everything we know won't work locally so the logs are readable
cred = DefaultAzureCredential(
    exclude_managed_identity_credential=True,
    exclude_workload_identity_credential=True,
    exclude_cli_credential=True,
    exclude_visual_studio_code_credential=True,
    exclude_powershell_credential=True,
    exclude_developer_cli_credential=True,
    exclude_shared_token_cache_credential=True,
)
print('DefaultAzureCredential chain configured. First source it\'ll try: EnvironmentCredential (because we set env vars).')

> We don't actually `get_token()` against real Entra here because that would require an Azure tenant. The point is: **same code**. In real Azure you'd call `cred.get_token('api://api-b/.default')` and get back a real token from Entra; the FastAPI `Authorization: Bearer` call that follows is identical.

## Wiring `DefaultAzureCredential` into a FastAPI client

Typical snippet you'd drop into `api-a` (or any caller):

```python
from azure.identity import DefaultAzureCredential
import httpx

_cred = DefaultAzureCredential()
_scope = 'api://api-b/.default'

def call_api_b(path: str) -> dict:
    token = _cred.get_token(_scope).token
    r = httpx.get(f'{API_B_URL}{path}', headers={'Authorization': f'Bearer {token}'}, timeout=5)
    r.raise_for_status()
    return r.json()
```

`cred.get_token()` is cached internally — it only hits Entra when the cached token is near expiry. Safe to call on every request.

## Things that bite people

- **Multi-tenant local dev**: `az login` gives you a token for your *default* tenant. Use `az login --tenant <id>` or `AZURE_TENANT_ID` env var.
- **Stale token cache**: if you rotate a secret and DefaultAzureCredential keeps using the old one, kill any local `msal` / `azure-cli` token caches in `~/.azure/` and `~/.IdentityService/`.
- **VS Code → az CLI mismatch**: disable `VisualStudioCodeCredential` explicitly unless you're sure which sign-in you want to use.
- **Logs hide the real error**: DefaultAzureCredential *only* tells you the last error. Enable debug logging (see cell above) to see all chain attempts.
- **Scopes vs resources**: v1 endpoints use `resource=<uri>`, v2 uses `scope=<uri>/.default`. Mixing them gives cryptic 400s.

## CI/CD without secrets — workload identity federation

Instead of storing `AZURE_CLIENT_SECRET` in GitHub Actions, you can configure federated credentials: GitHub mints a short-lived OIDC JWT, Entra trusts it, you get a token. Secret-less CI.

```yaml
- uses: azure/login@v2
  with:
    client-id: ${{ secrets.AZURE_CLIENT_ID }}
    tenant-id: ${{ secrets.AZURE_TENANT_ID }}
    subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}
    # no client-secret!
```

## Summary

- `DefaultAzureCredential` = one credential for all environments.
- Local: `az login` (Pattern A) or env vars (Pattern B).
- CI: federated identity, no secrets in vaults.
- Azure: managed identity, no secrets anywhere.
- Always enable debug logging when debugging credential issues.